# DeepSeek风格完整模型
## DeepSeek-style Complete Model

<img src="../images/logo.png" width=150>

DeepSeek结合了多种先进技术：RoPE位置编码、MoE专家混合、Sliding Window Attention和量化。本notebook整合所有技术，构建一个完整的DeepSeek风格模型。

DeepSeek combines multiple advanced technologies: RoPE positional encoding, MoE expert mixing, Sliding Window Attention, and quantization. This notebook integrates all technologies to build a complete DeepSeek-style model.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

# 导入所有组件 / Import all components
from typing import Optional, Tuple

class DeepSeekConfig:
    """DeepSeek配置 / DeepSeek Configuration"""
    def __init__(
        self,
        vocab_size=32000,
        embed_dim=512,
        num_heads=8,
        num_layers=4,
        max_seq_len=2048,
        num_experts=8,
        expert_ff_dim=2048,
        top_k=2,
        window_size=256,
        quantize_bits=4
    ):
        self.vocab_size = vocab_size
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.num_layers = num_layers
        self.max_seq_len = max_seq_len
        self.num_experts = num_experts
        self.expert_ff_dim = expert_ff_dim
        self.top_k = top_k
        self.window_size = window_size
        self.quantize_bits = quantize_bits

config = DeepSeekConfig()
print("DeepSeek Configuration:")
print(f"  Vocab: {config.vocab_size}, Embed: {config.embed_dim}, Heads: {config.num_heads}")
print(f"  Layers: {config.num_layers}, Experts: {config.num_experts}, Top-K: {config.top_k}")
print(f"  Window: {config.window_size}, Quantize: {config.quantize_bits}bit")

# RoPE + MoE + Sliding Window 完整实现
## Complete RoPE + MoE + Sliding Window Implementation

In [ ]:
def rotate_half(x):
    """标准RoPE的辅助函数：将张量后一半维度旋转到前一半"""
    x1, x2 = x[..., :x.shape[-1]//2], x[..., x.shape[-1]//2:]
    return torch.cat((-x2, x1), dim=-1)

# 完整的RoPE实现（精简版）/ Complete RoPE implementation (simplified)
class RotaryPositionalEmbedding(nn.Module):
    def __init__(self, dim, max_seq_len=2048, base=10000):
        super().__init__()
        inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2).float() / dim))
        t = torch.arange(max_seq_len)
        freqs = torch.outer(t, inv_freq)
        self.register_buffer('cos_cached', freqs.cos())
        self.register_buffer('sin_cached', freqs.sin())

    def forward(self, seq_len):
        return self.cos_cached[:seq_len], self.sin_cached[:seq_len]

def apply_rotary_pos_emb(q, k, cos, sin):
    """
    应用RoPE到Q和K
    标准RoPE：对相邻维度配对 (0,1), (2,3), (4,5), ... 进行旋转
    q, k: (batch, heads, seq, head_dim)
    cos, sin: (seq, head_dim/2)
    """
    if cos.dim() == 2:
        cos = cos.unsqueeze(0).unsqueeze(0)  # (1, 1, seq, head_dim/2)
        sin = sin.unsqueeze(0).unsqueeze(0)

    # 标准RoPE公式: q_rot = q * cos + rotate_half(q) * sin
    q_rot = q * cos + rotate_half(q) * sin
    k_rot = k * cos + rotate_half(k) * sin
    return q_rot, k_rot

# DeepSeek Transformer块
## DeepSeek Transformer Block

In [ ]:
class DeepSeekBlock(nn.Module):
    """
    DeepSeek Transformer块
    - 使用Sliding Window Attention + RoPE
    - MoE前馈网络
    
    DeepSeek Transformer block
    - Sliding Window Attention + RoPE
    - MoE feed-forward network
    """
    def __init__(self, config):
        super().__init__()
        self.ln1 = nn.LayerNorm(config.embed_dim)
        self.ln2 = nn.LayerNorm(config.embed_dim)
        
        # 交替使用滑动窗口和标准注意力
        # Alternate between sliding window and standard attention
        self.attention = SlidingWindowAttentionWithRoPE(
            config.embed_dim,
            config.num_heads,
            config.window_size
        )
        
        # MoE前馈 / MoE feed-forward
        self.moe = MoELayer(
            config.embed_dim,
            config.num_experts,
            config.expert_ff_dim,
            config.top_k
        )
        
        self.dropout = nn.Dropout(0.1)
    
    def forward(self, x, mask=None):
        # 自注意力 + RoPE
        x = x + self.dropout(self.attention(self.ln1(x), mask))
        # MoE前馈
        x = x + self.moe(self.ln2(x))
        return x

# 创建DeepSeek模型 / Create DeepSeek model
class DeepSeekModel(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        
        self.token_embedding = nn.Embedding(config.vocab_size, config.embed_dim)
        
        self.blocks = nn.ModuleList([
            DeepSeekBlock(config)
            for _ in range(config.num_layers)
        ])
        
        self.ln_f = nn.LayerNorm(config.embed_dim)
        self.head = nn.Linear(config.embed_dim, config.vocab_size, bias=False)
        
        self.apply(self._init_weights)
    
    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
    
    def forward(self, x, mask=None):
        x = self.token_embedding(x)
        for block in self.blocks:
            x = block(x, mask)
        return self.head(self.ln_f(x))

# 实例化模型 / Instantiate model
config = DeepSeekConfig(
    vocab_size=32000,
    embed_dim=512,
    num_heads=8,
    num_layers=4,
    max_seq_len=2048,
    num_experts=8,
    expert_ff_dim=2048,
    top_k=2,
    window_size=256
)

model = DeepSeekModel(config)
total_params = sum(p.numel() for p in model.parameters())
print(f"\nDeepSeek-style Model:")
print(f"  Total parameters: {total_params / 1e6:.1f}M")

# 估算激活参数 / Estimate active parameters
active_per_layer = (
    config.embed_dim * config.embed_dim * 3 +  # attention QKV
    config.embed_dim * config.embed_dim +      # attention output
    config.num_experts * config.top_k * config.expert_ff_dim * config.embed_dim / config.num_experts  # top-k experts
)
total_active = active_per_layer * config.num_layers
print(f"  Active parameters per forward: {total_active / 1e6:.1f}M")
print(f"  Sparsity ratio: {total_active / total_params * 100:.1f}%")

# 量化模块
## Quantization Module

In [ ]:
class Quantizer:
    def __init__(self, bits=4):
        self.bits = bits
        self.qmin = -(2 ** (bits - 1))
        self.qmax = 2 ** (bits - 1) - 1
    
    def quantize(self, x):
        scale = x.abs().max() / self.qmax
        x_quant = torch.round(x / scale).clamp(self.qmin, self.qmax)
        return x_quant.to(torch.int8), scale
    
    def dequantize(self, x_quant, scale):
        return x_quant.float() * scale

class QuantizedDeepSeekModel:
    """量化版本的DeepSeek模型"""
    
    def __init__(self, model, bits=4):
        self.model = model
        self.bits = bits
        self.quantizer = Quantizer(bits)
        self.quantized_layers = {}
    
    def quantize(self):
        """量化所有线性层 / Quantize all linear layers"""
        print(f"Quantizing model to {self.bits} bits...")
        
        for name, module in self.model.named_modules():
            if isinstance(module, nn.Linear):
                quantized_weight, scale = self.quantizer.quantize(module.weight.data)
                self.quantized_layers[name] = {
                    'weight': quantized_weight,
                    'scale': scale,
                    'bias': module.bias.data if module.bias is not None else None
                }
        
        print(f"  Quantized {len(self.quantized_layers)} layers")
    
    def get_model_size(self):
        """计算模型大小 / Calculate model size"""
        fp32_size = sum(p.numel() * 4 for p in self.model.parameters()) / (1024 ** 2)
        
        quantized_bytes = sum(
            w['weight'].numel() * (self.bits / 8) + 4  # weight + scale
            for w in self.quantized_layers.values()
        )
        quant_size = quantized_bytes / (1024 ** 2)
        
        return fp32_size, quant_size

# 测试量化 / Test quantization
quantized_model = QuantizedDeepSeekModel(model, bits=4)
quantized_model.quantize()

fp32_size, quant_size = quantized_model.get_model_size()
print(f"\nModel size:")
print(f"  FP32: {fp32_size:.1f} MB")
print(f"  {quantized_model.bits}-bit: {quant_size:.1f} MB")
print(f"  Compression: {fp32_size / quant_size:.1f}x")

# 前向传播测试
## Forward Pass Test

In [ ]:
import time

# 测试完整前向 / Test complete forward
x = torch.randint(0, config.vocab_size, (2, 128))

model.eval()
with torch.no_grad():
    start = time.time()
    output = model(x)
    elapsed = time.time() - start

print(f"Forward pass test:")
print(f"  Input: {x.shape}")
print(f"  Output: {output.shape}")
print(f"  Time: {elapsed*1000:.2f} ms")
print(f"  Throughput: {2 * 128 / elapsed:.0f} tokens/sec")

# 测试不同序列长度 / Test different sequence lengths
print("\nSequence length scaling:")
for seq_len in [64, 128, 256, 512]:
    x = torch.randint(0, config.vocab_size, (1, seq_len))
    with torch.no_grad():
        start = time.time()
        output = model(x)
        elapsed = time.time() - start
    print(f"  seq_len={seq_len}: {elapsed*1000:.1f} ms, {seq_len/elapsed:.0f} tokens/sec")

# 架构可视化
## Architecture Visualization

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Visualize DeepSeek architecture
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# 1. MoE Expert routing visualization
ax1 = axes[0]
experts = ['Expert 1', 'Expert 2', 'Expert 3', 'Expert 4', 'Expert 5', 'Expert 6', 'Expert 7', 'Expert 8']
load_balance = np.random.dirichlet([0.3]*8)
tokens_perExpert = load_balance * 1000
colors = plt.cm.Blues(np.linspace(0.4, 0.9, 8))
bars = ax1.bar(experts, tokens_perExpert, color=colors)
ax1.set_xlabel('Expert')
ax1.set_ylabel('Tokens Processed')
ax1.set_title('MoE Expert Load Distribution (Top-K=2)')
ax1.tick_params(axis='x', rotation=45)

# 2. Sliding Window Attention pattern
ax2 = axes[1]
seq_len = 64
window_size = 16
attn_pattern = np.zeros((seq_len, seq_len))
for i in range(seq_len):
    start = max(0, i - window_size // 2)
    end = min(seq_len, i + window_size // 2 + 1)
    attn_pattern[i, start:end] = 1
im = ax2.imshow(attn_pattern[:32, :32], cmap='Blues', aspect='auto')
ax2.set_xlabel('Key Position')
ax2.set_ylabel('Query Position')
ax2.set_title('Sliding Window Attention Pattern (w=16)')

# 3. Quantization effect
ax3 = axes[2]
bits = ['FP32', 'INT8', 'INT4']
sizes = [total_params * 4 / 1e6, total_params * 1 / 1e6, total_params * 0.5 / 1e6]
colors = ['#3498db', '#2ecc71', '#e74c3c']
bars = ax3.bar(bits, sizes, color=colors)
ax3.set_ylabel('Model Size (MB)')
ax3.set_title('Quantization压缩效果 / Quantization Effect')
for bar, size in zip(bars, sizes):
    ax3.text(bar.get_x() + bar.get_width()/2., bar.get_height(),
             f'{size:.1f}MB', ha='center', va='bottom')

plt.tight_layout()
plt.savefig('../images/deepseek_architecture.png', dpi=150, bbox_inches='tight')
plt.show()

print("DeepSeek Architecture Visualization saved!")

# 完整技术栈总结
## Complete Technology Stack Summary

In [ ]:
print("\n" + "="*60)
print("DeepSeek-Style Model: Complete Technology Stack")
print("="*60)

tech_stack = [
    ("BPE Tokenizer", "GPT-2", "子词分词，vocab=32k"),
    ("Token Embedding", "Standard", "可学习嵌入"),
    ("RoPE", "LLaMA/DeepSeek", "旋转位置编码，支持长上下文"),
    ("Sliding Window Attn", "Longformer", "O(n×w)复杂度"),
    ("MoE", "Switch/DeepSeek", "8专家top-2激活"),
    ("Quantization", "GPTQ/AWQ", "INT4权重量化"),
]

print(f"\n{'技术':<20} {'来源':<15} {'描述':<30}")
print("-"*60)
for tech, source, desc in tech_stack:
    print(f"{tech:<20} {source:<15} {desc:<30}")

print("\n" + "="*60)
print("模型架构参数")
print("="*60)
print(f"  Vocab size: {config.vocab_size}")
print(f"  Embed dim: {config.embed_dim}")
print(f"  Num heads: {config.num_heads}")
print(f"  Num layers: {config.num_layers}")
print(f"  Num experts: {config.num_experts}")
print(f"  Expert FF dim: {config.expert_ff_dim}")
print(f"  Top-K: {config.top_k}")
print(f"  Window size: {config.window_size}")
print(f"  Max seq len: {config.max_seq_len}")

print(f"\nTotal parameters: {total_params / 1e6:.1f}M")
print(f"Active parameters: {total_active / 1e6:.1f}M")
print(f"Sparsity: {total_active / total_params * 100:.1f}%")

In [ ]:
# DeepSeek技术栈流水线 / DeepSeek Pipeline Visualization
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1. Full pipeline flow
ax1 = axes[0]
ax1.axis('off')

stages = [
    ('Input', 'Tokenize
(BPE)'),
    ('Embedding', 'Token + Position
(RoPE)'),
    ('Attention', 'Sliding Window
+ RoPE'),
    ('FFN', 'MoE
(8 experts, top-2)'),
    ('Quantize', 'INT4
Weights'),
    ('Output', 'Generated
Tokens')
]

colors = ['#3498db', '#2ecc71', '#e74c3c', '#9b59b6', '#f39c12', '#1abc9c']
for i, (stage, desc) in enumerate(stages):
    box = plt.Rectangle((i*1.5, 0), 1.3, 1, facecolor=colors[i], edgecolor='black', linewidth=2)
    ax1.add_patch(box)
    ax1.text(i*1.5 + 0.65, 0.7, stage, ha='center', va='center', fontsize=9, fontweight='bold')
    ax1.text(i*1.5 + 0.65, 0.3, desc, ha='center', va='center', fontsize=8)

for i in range(len(stages)-1):
    ax1.annotate('', xy=((i+1)*1.5, 0.5), xytext=(i*1.5+1.3, 0.5),
                arrowprops=dict(arrowstyle='->', color='gray', lw=2))

ax1.set_xlim(-0.2, len(stages)*1.5 + 0.5)
ax1.set_ylim(-0.3, 1.5)
ax1.set_title('DeepSeek Pipeline
(From Input to Output)', fontsize=12)

# 2. Parameter efficiency
ax2 = axes[1]
labels = ['Dense
(all params)', 'MoE
(8 experts)', 'MoE + Quant
(INT4)']
x = np.arange(len(labels))
memory_gb = [350, 175, 44]
colors = ['#3498db', '#e74c3c', '#2ecc71']

bars = ax2.bar(x, memory_gb, color=colors)
ax2.set_ylabel('Memory Required (GB)')
ax2.set_title('DeepSeek Memory Efficiency
(175B Parameter Model)')
ax2.set_xticks(x)
ax2.set_xticklabels(labels)

for bar, mem in zip(bars, memory_gb):
    ax2.text(bar.get_x() + bar.get_width()/2., bar.get_height(),
             f'{mem}GB', ha='center', va='bottom', fontsize=10)

ax2.annotate('87.5% reduction
with MoE + Quant', 
             xy=(2, 44), xytext=(1.5, 120),
             arrowprops=dict(arrowstyle='->', color='gray'),
             fontsize=9, color='gray')

plt.tight_layout()
plt.savefig('../images/deepseek_pipeline.png', dpi=150, bbox_inches='tight')
plt.show()

print("DeepSeek pipeline visualization saved!")

# 已实现 / Implemented

本notebook已完整实现以下内容：

1. **RoPE + MoE + Sliding Window** - 三大技术整合
2. **DeepSeek Transformer块** - 完整块实现
3. **量化模块** - INT4量化
4. **前向传播测试** - 不同序列长度性能
5. **架构可视化** - MoE负载、注意力模式、量化效果

## 扩展阅读 / Further Reading

| 主题 | 说明 | 推荐资源 |
|------|------|----------|
| **分布式训练** | ZeRO/FSDP策略 | [DeepSpeed](https://www.deepspeed.ai/) |
| **长上下文优化** | YaRN/NTK外推 | [DeepSeek-V4 Tech Report](https://arxiv.org/abs/2401.14166) |
| **推理优化** | Flash/Paged Attention | [vLLM](https://github.com/vllm-project/vllm) |
| **持续预训练** | SFT/RLHF/DPO对齐 | [LLaMA Factory](https://github.com/hiyouga/LLaMA-Factory) |
